# Mount Google Drive + Install Libraries

In [1]:
# Install required libraries (PyTorch, sklearn, matplotlib)
!pip install torch torchvision scikit-learn matplotlib PyQt5 ultralytics --quiet 


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install seaborn



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install seaborn ultralytics


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Libraries

In [4]:
import os  # This library is used to work with folders and file paths, for example, to open a data folder or check if a file exists.
import torch   # This is the main library for deep learning. Through it, we use tensors and run the model on the GPU.
import torch.nn as nn   # This part contains the neural network layers , such as Linear, Conv2D, and loss functions. Through it, we build the model architecture.
import torch.optim as optim   # Provides optimization algorithms like Adam and SGD
from torchvision import datasets, transforms, models  # Tools for image datasets, preprocessing, and pretrained models
#This part is related to images: datasets → to load images from folders, transforms → to modify images (resize, normalize, etc.) ,models → pre-trained models such as ResNet
from torch.utils.data import DataLoader, random_split  # Data loading and dataset splitting utilities , DataLoader → splits images into batches during training
# random_split → splits the data into training, validation, and testing sets
from sklearn.metrics import classification_report    # Used to evaluate the model’s performance after testing. It gives us Precision, Recall, and F1-score in an organized way.
import cv2
import numpy as np
from PIL import Image , ImageFile
import json 
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

# How to handle data in PyTorch.

In [5]:
from torch.utils.data import Dataset

# By creating a new "class" named DatasetWithTransform.
class DatasetWithTransform(Dataset):
    # Creating an object from the class.
    # The code takes a list of modifications (such as resizing, rotating, or converting the image to Tensor) and stores them.
    # We set `None` as the default value so that the code will work even if we don't want to apply any modifications.
    def __init__(self, dataset, transform=None):
        # It takes the raw data set and stores it.
        self.dataset = dataset
        self.transform = transform

    # This function is requested by PyTorch to determine the total number of images in the project.
    def __len__(self):
      # Simply restore the length of the original dataset.
        return len(self.dataset)

    # This function is called every time the model needs an image for training.
    #idx (image number or index).
    def __getitem__(self, idx):
      # It goes to the original dataset and retrieves the image (img) and its label based on the required number.
        img, label = self.dataset[idx]
        if self.transform:
          # The image is passed through a "filter" or adjustments (such as Resize or ToTensor) and the image is updated with the modified version.
            img = self.transform(img)
        return img, label

In [6]:
class ApplyCLAHE(object):
  # clip_limit: Controls the "contrast". If it increases, the image becomes sharper, and if it decreases, it becomes softer.
  # tile_grid_size: Dividing the image into small squares (8x8) to process it piece by piece, this is what makes it "Adaptive".
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        # Convert the image from PIL format (for display) to a Numpy number array so that we can perform calculations on it.
        img_np = np.array(img)

        # Are the image colored.
        if len(img_np.shape) == 3:
            # Convert the image to a color system called LAB.
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            # Separating the image into three channels.
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            cl = clahe.apply(l)
            # Re-merging the channels together after improving the lighting.
            limg = cv2.merge((cl, a, b))
            final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        else:
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tile_grid_size=self.tile_grid_size)
            final_img = clahe.apply(img_np)

        return Image.fromarray(final_img)

In [7]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomGrayscale(p=0.5),
    # Converting the image from (Pillow Image) format to (Tensor); that is, converting it to a matrix of numbers that the graphics card (GPU) can understand.
    transforms.ToTensor(),
    # This involves subtracting the mean and dividing by the standard deviation. This brings the pixel values ​​closer to zero, greatly speeding up the model learning process.
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # Increasing clip_limit to 3.0 makes details stand out more strongly than in 2.0.
    ApplyCLAHE(clip_limit=3.0, tile_grid_size=(8, 8)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Select Device (GPU/CPU)


In [8]:
# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cpu


# Image Transformations

In [9]:
from torchvision import transforms  # Provides image preprocessing utilities , We use it to modify the image so that it fits the neural network.

val_test_transform = transforms.Compose([ # Combine multiple image transformations into one pipeline
    transforms.Resize((224, 224)),  # Resize image to 224x224 (ResNet input size)
    transforms.Lambda(lambda img: img.convert("RGB")),  # Convert grayscale images to RGB (3 channels)
   # Some X-ray images are grayscale (single channel).
   # So we need to convert the image to 3 channels even if it is grayscale.

    transforms.ToTensor(), # Convert image to PyTorch tensor, PyTorch only works with tensors.
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean values
        std=[0.229, 0.224, 0.225]  # ImageNet standard deviation values
    )
])

In [10]:
from PIL import ImageFile # Import module to handle image file loading
"This is the PIL library, which is responsible for opening and handling images in Python. We need it to safely read X-ray images and perform preprocessing on them."
ImageFile.LOAD_TRUNCATED_IMAGES = True # Allow loading of images that are partially corrupted or truncated

# Load Dataset

In [11]:
data_dir = r"C:\Users\MSI-H\OneDrive\Desktop\X-Ray"   # Path to your dataset folder, adjust as needed

dataset = datasets.ImageFolder(root=data_dir)  # Load images without applying transformations initially

class_names = dataset.classes # Get list of class names, retrieves the names of all the classes present in the datase
num_classes = len(class_names)  # Count the number of classes

print("Classes:", class_names)   # Print names of classes , We print the class names to verify that the dataset was read correctly.
print("Total images:", len(dataset))   # Print total number of images , To check the size of the data before starting training.
print("Num classes:", num_classes)  # Print total number of classes

Classes: ['Not_X_ray image', 'injury', 'normal']
Total images: 8803
Num classes: 3


# Split Dataset

In [12]:
total_size = len(dataset)   # Get total number of images in the dataset

train_size = int(0.7 * total_size) # 70% of data for training
val_size   = int(0.15 * total_size)  # 15% of data for validation , This set is used to evaluate the model’s performance during training and to help improve it.

test_size  = total_size - train_size - val_size # Remaining 15% for testing , The remaining data (approximately 15%) is used for the model’s final testing after training.


# Split the dataset into raw subsets
raw_train_dataset, raw_val_dataset, raw_test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

# Apply transformations using the DatasetWithTransform wrapper
train_dataset = DatasetWithTransform(raw_train_dataset, transform=train_transforms)
val_dataset = DatasetWithTransform(raw_val_dataset, transform=val_test_transform)
test_dataset = DatasetWithTransform(raw_test_dataset, transform=val_test_transform)


print(f"Train: {len(train_dataset)}")  # Print number of training images
print(f"Validation: {len(val_dataset)}")  # Print number of validation images
print(f"Test: {len(test_dataset)}") # Print number of test images

Train: 6162
Validation: 1320
Test: 1321


# DataLoader

In [13]:
batch_size = 32   # Number of images per batch , The batch size affects the training speed and memory usage.

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # Create DataLoader for training set
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)# Create DataLoader for validation set
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # Create DataLoader for test set


# Load Pretrained DenseNet121

In [14]:
import torch
import torch.nn as nn
from torchvision import models

# Load the DenseNet121 model.
# We told the program not only to carry the structure, but also the weights (experience) the model gained from training on the popular ImageNet database.
# This means the model already knows how to distinguish lines, edges, and shapes.
model = models.densenet121(weights='IMAGENET1K_V1')

# Freeze the base layers (to preserve the features extracted from ImageNet).
for param in model.parameters():
    param.requires_grad = False


# The number of features in DenseNet121 is always 1024
num_ftrs = model.classifier.in_features

model.classifier = nn.Sequential(
    # A layer receives 1024 features and compresses them to become 512.
    nn.Linear(num_ftrs, 512),
    # The Activation Function helps the model understand complex and non-linear relationships.
    nn.ReLU(),
    # To prevent data from overfitting.
    nn.Dropout(0.5),
    # An additional layer to increase the depth of understanding and reduce the data to 256.
    nn.Linear(512, 256),
    nn.ReLU(),
    # The crucial final layer. The 256 features are reduced to just 3 outputs.
    nn.Linear(256, 3)
)

model = model.to(device)

# Loss & Optimizer


In [15]:
# Addressing data imbalances (Class Weights).
# 2.5 for not_xray_image :"If you misclassify a non-X-ray image, I will penalize you two and a half times more than if you misclassify anything else".
# This forces the model to pay extra attention to this category.
weights = torch.tensor([1.0, 1.0, 2.5]).to(device)
# This is the "loss function". By adding weights to it, the model becomes very "sensitive" to weak or hard class.
criterion = nn.CrossEntropyLoss(weight=weights)

# Partial defrosting (Unfreezing Layers)
# Unfreezing layer4.
for param in model.features.denseblock4.parameters():
    # Allowing weights in these layers to change and learn.
    param.requires_grad = True


optimizer = torch.optim.Adam([
    # Deep layers (denseblock4) have a very low learning rate (0.00001).
    # The reason: Because these layers have strong prior knowledge, we want to modify them very slowly so they don't lose what they've learned (causing catastrophic corruption).
    {'params': model.features.denseblock4.parameters(), 'lr': 1e-5},
    # The last layer (classifier) ​​requires a learning rate ten times higher (0.001).
    # The reason: This layer is completely new and needs to be learned "quickly" to understand the difference between the three classes in your project.
    {'params': model.classifier.parameters(), 'lr': 1e-3}
])

# Training Function

In [16]:
train_losses = []    # List to store training loss for each epoch
val_losses = []  # List to store validation loss for each epoch
train_accuracies = []    # Training accuracy per epoch
val_accuracies = []      # Validation accuracy per epoch

def train_model(model, train_loader, val_loader, epochs=10):# Define training function
    for epoch in range(epochs):# Loop over epochs
        # ===== Training =====
        model.train() # Set model to training mode
        train_loss = 0 # Initialize cumulative training loss for this epoch
        correct_train = 0
        total_train = 0

        for images, labels in train_loader:# Loop over batches
            images, labels = images.to(device), labels.to(device) # Move to GPU/CPU

            optimizer.zero_grad() # Clear previous gradients
            outputs = model(images) # Forward pass
            loss = criterion(outputs, labels) # Compute loss
            loss.backward() # Backpropagation
            optimizer.step() # Update weights

            train_loss += loss.item() # Accumulate batch loss

            # Accuracy calculation
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        train_loss /= len(train_loader) # Compute average training loss for the epoch
        train_accuracy = correct_train / total_train

        train_losses.append(train_loss) # Store it
        train_accuracies.append(train_accuracy)

        # ===== Validation =====
        model.eval() # Set model to evaluation mode
        val_loss = 0
        correct_val = 0
        total_val = 0

        with torch.no_grad(): # Disable gradient computation
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        val_loss /= len(val_loader) # Average validation loss
        val_accuracy = correct_val / total_val

        val_losses.append(val_loss) # Store it
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}"
        )
# We print the loss for each epoch to monitor training in real time. This helps

In [ ]:
train_model(model, train_loader, val_loader, epochs=20) # Start training for 18 epochs
# The number of epochs is 18, meaning the network will go through all the dataset


Epoch 1/20 | Train Loss: 0.5778, Train Acc: 0.6999 | Val Loss: 0.4369, Val Acc: 0.8326
Epoch 2/20 | Train Loss: 0.3666, Train Acc: 0.8442 | Val Loss: 0.3694, Val Acc: 0.8780
Epoch 3/20 | Train Loss: 0.2933, Train Acc: 0.8858 | Val Loss: 0.2596, Val Acc: 0.9061
Epoch 4/20 | Train Loss: 0.2320, Train Acc: 0.9085 | Val Loss: 0.2112, Val Acc: 0.9212
Epoch 5/20 | Train Loss: 0.2004, Train Acc: 0.9226 | Val Loss: 0.1977, Val Acc: 0.9121
Epoch 6/20 | Train Loss: 0.1942, Train Acc: 0.9278 | Val Loss: 0.1651, Val Acc: 0.9379
Epoch 7/20 | Train Loss: 0.1568, Train Acc: 0.9429 | Val Loss: 0.1367, Val Acc: 0.9348


# Plot Loss

import matplotlib.pyplot as plt # Import matplotlib for plotting graphs

plt.plot(train_losses, label="Train Loss")  # Plot training loss
plt.plot(val_losses, label="Validation Loss") # Plot validation loss
plt.xlabel("Epochs")  # Label for x-axis
plt.ylabel("Loss")  # Label for y-axis
plt.legend()   # Show legend
plt.title("Training & Validation Loss") # Add title to the plot
plt.show()  # Display the plot

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(train_accuracies) + 1)

plt.figure()
plt.plot(epochs_range, train_accuracies)
plt.plot(epochs_range, val_accuracies)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training & Validation Accuracy")
plt.legend(["Train Accuracy", "Validation Accuracy"])
plt.show()

# Classification Report + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix # Import metrics for evaluation
import seaborn as sns  # Import seaborn for better visualization of confusion matrix
import numpy as np # Import numpy for numerical operations

In [ ]:
model.eval() # Set model to evaluation mode
y_true, y_pred = [], [] # Lists to store true and predicted labels

with torch.no_grad(): # Disable gradient computation
    for images, labels in test_loader: # Loop over test batches
        images, labels = images.to(device), labels.to(device) # Move to GPU/CPU
        outputs = model(images) # Forward pass
        _, preds = torch.max(outputs, 1)  # Get predicted class index

        y_true.extend(labels.cpu().numpy())   # Store true labels
        y_pred.extend(preds.cpu().numpy())   # Store predicted labels

# Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names)) # Print detailed classification metrics

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)  # Compute confusion matrix

plt.figure(figsize=(6,5))  # Set figure size

sns.heatmap(cm, annot=True, fmt="d",            # Plot confusion matrix as a heatmap
            xticklabels=class_names,           # X-axis labels
            yticklabels=class_names,           # Y-axis labels
            cmap="Blues")                      # Color map
plt.xlabel("Predicted")  # Label for x-axis
plt.ylabel("Actual")     # Label for y-axis
plt.title("Confusion Matrix")  # Add title
plt.show()  # Display the confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision import transforms

# Grad CAM Class
class GradCAM:
  # Initialization function (__init__)
    def __init__(self, model, target_layer):
        # Receiving and storing the (DenseNet121) model.
        self.model = model
        # Identifying the "target layer" from which we will take the visual maps (often the last convolutional layer before classification).
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        # This is a "spy" function that asks PyTorch: "When the image is forwarded (Forward Pass), save for me what this layer saw in the activations variable."
        self.target_layer.register_forward_hook(self.save_activation)
        # A reverse "spy" function: "When the model goes back to calculate the error (Backward Pass), save for me the magnitude of the gradients or slope in the variable gradients."
        self.target_layer.register_backward_hook(self.save_gradient)

    # It stores "visual maps" (what the layer saw).
    def save_activation(self, module, input, output):
        self.activations = output

    # It stores "tendencies" (what is the most influential part in the final result).
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image, class_idx=None):
        output = self.model(input_image)

        if class_idx is None:
            # If we do not specify a particular class, the code will automatically select the class with the highest probability.
            class_idx = torch.argmax(output)

        self.model.zero_grad()
        # Identify the value we want to interpret.
        loss = output[0, class_idx]
        # Starting with the reverse calculation process to extract the weights and slopes.
        loss.backward()

        gradients = self.gradients[0].cpu().data.numpy()
        activations = self.activations[0].cpu().data.numpy()

        # The "average importance" is calculated for each channel in the layer.
        # A channel with a high slope means it contains important information about the fracture.
        weights = np.mean(gradients, axis=(1, 2))

        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for i, w in enumerate(weights):
            # Here we multiply each "visual map" by its "weight of importance" and add them together. The result is a map showing the most influential places.
            cam += w * activations[i]
        # The technique is called (ReLU) on the map; where we keep only the areas that have a positive impact on the decision and delete the areas that have no relation (negative values).
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (224, 224))
        cam = cam - cam.min()
        # (Normalization) To make the map values ​​between 0 and 1, which makes it easier to color them later (red for the strongest and blue for the weakest).
        cam = cam / cam.max()
        return cam

# Severity Assessment (Logic)

In [ ]:
def cam_severity_score(cam):
  # count pixels with high activation
  injury_area = (cam > 0.6).sum()
  total_area = cam.size

  ratio = injury_area / total_area
  return ratio

def severity_assessment(confidence, cam):

    area_ratio = cam_severity_score(cam)

    score = (0.6 * confidence) + (0.4 * area_ratio)

    if score < 0.4:
        return "Mild"
    elif score < 0.7:
        return "Moderate"
    else:
        return "Severe"

In [ ]:
def get_medical_advice(predicted_class, severity, confidence):

    if predicted_class == "injury" and confidence > 0.6:
        # Corrected severity check from "HIGH" to "Severe" to match severity_assessment output
        if severity == "Severe":
            return "⚠️ Severe injury: Immobilize the injured limb immediately and go to the emergency room."
        elif severity == "Moderate":
            return "🚨 Moderate injury: A possible fracture appears. Please do not move the limb and consult a doctor for a splint."
        else: # Implies severity == "Mild"
            return "🩹 Minor injury: Additional X-rays or consultation with a specialist are recommended."
    else:
        THRESHOLD = 0.75

        if confidence < THRESHOLD:
            return("The result is uncertain: please consult a human doctor immediately; the model cannot be certain.")
        else:
            # Added an explicit return for cases where it's not an injury
            # and confidence is high (e.g., 'normal' or 'Not_X_ray image' with high confidence).
            return "No specific medical advice for this classification and confidence level. Consult a medical professional for further assessment if needed."

    # The recursive call and print statement below caused the RecursionError and must be removed.
    # advice = get_medical_advice(predicted_class, severity, confidence)
    # print(f"First Aid: {advice}")



 # Now you can upload your image:

In [ ]:
from PIL import Image  # import image handling library
from PyQt5.QtWidgets import QApplication, QFileDialog  # import GUI tools for file selection
import sys  # system module for running the app


app = QApplication.instance() or QApplication(sys.argv)  
# use existing app if it exists, otherwise create a new one

current_image_path, _ = QFileDialog.getOpenFileName(
    None,  # no parent window
    "Select X-Ray Image",  # window title
    "",  # default folder
    "Image Files (*.jpg *.jpeg *.png *.bmp *.tiff)"  # allowed image types
)

if not current_image_path:
    # if user didn't select any file
    raise ValueError("❌ No image selected. Please run again and select an image.")

print(f"✅ Image loaded: {current_image_path}")  # show selected image path

# DenseNet Inference + GradCAM

In [ ]:

gradcam = GradCAM(model, model.features.denseblock4)  
# initialize Grad-CAM using the model and last dense block layer

transform = val_test_transform  
# set the image transformation (preprocessing for validation/test)

# LOAD IMAGE 

fn = current_image_path  
# get image path selected by the user

image_pil = Image.open(fn).convert("RGB")  
# open image and convert it to RGB format

input_tensor = transform(image_pil).unsqueeze(0).to(device)  
# apply preprocessing, add batch dimension, and move to GPU/CPU

# ENABLE GRADCAM LAYER 

original_states = {}  
# dictionary to store original requires_grad states

for name, param in model.named_parameters():
    # loop through all model parameters

    if name.startswith('features.denseblock4'):
        # check only parameters from denseblock4 layer

        original_states[name] = param.requires_grad  
        # save original gradient state

        param.requires_grad = True  
        # force enable gradients for Grad-CAM visualization

# INFERENCE

model.eval()  
# set model to evaluation mode (disable dropout, batchnorm updates)

output = model(input_tensor)  
# forward pass (get raw predictions)

probs = torch.softmax(output, dim=1)  
# convert logits to probabilities

conf, pred = torch.max(probs, 1)  
# get highest probability and predicted class index

injury_name = class_names[pred.item()]  
# convert class index to class label name

print(f"Predicted Class : {injury_name}")  
# print predicted class name

print(f"Confidence      : {round(conf.item(), 4)}")  
# print confidence score rounded to 4 decimals


# YOLO Detection

In [ ]:
from ultralytics import YOLO
import json
import cv2
import matplotlib.pyplot as plt

model_path = r"C:\Users\MSI-H\OneDrive\Desktop\medray_final\medray_final\best.pt"
json_path = r"C:\Users\MSI-H\OneDrive\Desktop\medray_final\medray_final\bone_regions_full_fixed.json"

def get_bone_name(cx, cy, regions):
    for section in ["up", "down"]:
        for bone, coords in regions[section].items():
            x1, y1, x2, y2 = coords
            if x1 <= cx <= x2 and y1 <= cy <= y2:
                return bone
    

def get_severity_yolo(conf):
    if conf > 0.85:
        return "Severe"
    elif conf > 0.6:
        return "Moderate"
    else:
        return "Mild"

def run_yolo(image_path):
    image_cv = cv2.imread(image_path)
    if image_cv is None:
        raise ValueError("Could not load image for YOLO")

    original_cv = image_cv.copy()
    h, w = image_cv.shape[:2]

    model_yolo = YOLO(model_path)

    with open(json_path, "r") as f:
        bone_regions_full = json.load(f)

    side = "right" if "R" in image_path.upper() else "left"
    bone_regions = bone_regions_full[side]

    results = model_yolo.predict(image_cv, conf=0.6, iou=0.4, verbose=False)

    detections = []

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            conf_val = float(box.conf[0])
            if conf_val < 0.6:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            cx = ((x1 + x2) / 2) / w
            cy = ((y1 + y2) / 2) / h

            bone = get_bone_name(cx, cy, bone_regions)
            sev  = get_severity_yolo(conf_val)

            detections.append({
                "box": (x1, y1, x2, y2),
                "confidence": conf_val,
                "bone": bone,
                "severity": sev
            })

    detections = sorted(detections, key=lambda x: x["confidence"], reverse=True)[:3]

    image_boxes = original_cv.copy()

    color_map = {
        "Severe": (0,0,255),
        "Moderate": (0,255,255),
        "Mild": (0,255,0)
    }

    for det in detections:
        x1, y1, x2, y2 = det["box"]
        color = color_map[det["severity"]]

        cv2.rectangle(image_boxes, (x1,y1), (x2,y2), color, 3)

        label = f"{det['bone']} | {det['severity']} | {det['confidence']:.2f}"

        cv2.putText(image_boxes, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(image_boxes, cv2.COLOR_BGR2RGB))
    plt.title("YOLO - Top 3 Fractures (Confidence >= 60%)")
    plt.axis("off")
    plt.show()

    return detections

print("YOLO functions ready.")

# Final Report (DenseNet + YOLO)

In [ ]:
if injury_name in ("Not_X_ray image", "normal"):  
    # check if image is normal or not an X-ray (no injury case)

    # Normal / Not X-Ray — skip YOLO 

    plt.figure(figsize=(5, 5))  # set image display size
    plt.imshow(image_pil)  # show original image
    plt.title(f"Predicted: {injury_name} (Confidence: {conf.item():.2f})")  # show prediction result
    plt.axis("off"); plt.show()  # hide axes and display image

    print("\n FINAL REPORT")  # report title

    print(f"Predicted Class : {injury_name}")  # print predicted class
    print(f"Confidence      : {round(conf.item(), 4)}")  # print confidence score
    print(f"Severity        : -")  # no severity (normal case)
    print(f"First Aid       : -")  # no first aid needed
    print(f"Injury Location : No injury detected in the image you uploaded")  # no injury message

else:
    #  Injury detected → run GradCAM + YOLO

    cam = gradcam.generate(input_tensor)  
    # generate Grad-CAM heatmap

    severity = severity_assessment(conf.item(), cam)  
    # calculate severity based on confidence and heatmap

    aid = get_medical_advice(injury_name, severity, conf.item())  
    # get medical advice based on prediction

    img_np = np.array(image_pil.resize((224, 224)))  
    # convert image to numpy array and resize

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)  
    # convert CAM to colored heatmap

    superimposed = heatmap * 0.4 + img_np  
    # overlay heatmap on original image

    plt.figure(figsize=(10, 4))  # set figure size
    plt.subplot(1, 3, 1); 
    plt.imshow(img_np); 
    plt.title("Original"); 
    plt.axis("off")  
    # show original image

    plt.subplot(1, 3, 2); 
    plt.imshow(cam, cmap="jet"); 
    plt.title("GradCAM"); 
    plt.axis("off")  
    # show Grad-CAM heatmap

    plt.subplot(1, 3, 3); 
    plt.imshow(np.uint8(superimposed)); 
    plt.title("Overlay"); 
    plt.axis("off")  
    # show combined image (heatmap + original)

    plt.tight_layout(); 
    plt.show()  
    # adjust layout and display plots

    #  YOLO detection 

    detections = run_yolo(current_image_path)  
    # run YOLO model for fracture detection

    # FINAL REPORT 

    print(" FINAL REPORT")  # print report title

    print(f"Predicted Class : {injury_name}")  # predicted class
    print(f"Confidence      : {round(conf.item(), 4)}")  # confidence value
    print(f"Severity        : {severity}")  # severity result
    print(f"First Aid       : {aid}")  # medical advice

    if len(detections) == 0:  
        # if no fractures detected

        print("Injury Location : No fractures detected (confidence >= 60%)")  
        # message for no detection

    else:
        for i, det in enumerate(detections):  
            # loop over detected fractures

            print(f"Fracture {i+1}:")  # fracture number
            print(f"  - Bone        : {det['bone']}")  # bone name
            print(f"  - Confidence  : {det['confidence']:.2f}")  # confidence
            print(f"  - Severity    : {det['severity']}")  # severity level

    print("=" * 45)  # end report

    report = [{"densenet": {  # create final report structure
        "class": injury_name,  # predicted class
        "confidence": round(conf.item(), 4),  # confidence score
        "severity": severity,  # severity result
        "first_aid": aid  # medical advice
    }, "yolo_detections": detections}]  # add YOLO results

    with open("medical_report.json", "w") as f:  
        # open file to save report

        json.dump(report, f, indent=4)  
        # save report in JSON format

    print("Report saved to medical_report.json")  
    # confirm saving report


# RESTORE MODEL STATE 

for name, param in model.named_parameters():  
    # loop over all model parameters

    if name.startswith("features.denseblock4"):  
        # only target denseblock4 layer

        param.requires_grad = original_states[name]  
        # restore original gradient settings

In [ ]:
torch.save(model.state_dict(), 'best_model.pt')
print("Model saved ✓")